In [8]:
import pandas as pd
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import MERGED_DATA

def statement_to_dict_df(file, statement_name):
    # Extract the ticker from the filename by removing the statement name and file extension
    ticker = Path(file).stem.replace(f"_{statement_name}", "")

    df = pd.read_csv(file)

    metric_col = df.columns[0]

    rows = []

    # Iterate over each date column (starting from the second column) and create a dictionary of metrics for that date
    for date in df.columns[1:]:
        metrics = (
            df[[metric_col, date]]
            .dropna()
            .set_index(metric_col)[date]
            .to_dict()
        )

        rows.append({
            "company": ticker,
            "date": date,
            statement_name: metrics
        })

    return pd.DataFrame(rows)

In [9]:
from pathlib import Path

DATA_DIR = Path("../data")
BALANCE_SHEET_DIR = DATA_DIR / "balance_sheet"
CASH_FLOW_DIR = DATA_DIR / "cashflows"
FINANCIALS_DIR = DATA_DIR / "financials"

tickers = set()

for file in BALANCE_SHEET_DIR.glob("*.csv"):
    ticker = file.stem.replace("_balancesheet", "")
    tickers.add(ticker)

print(f"Found {len(tickers)} companies")

Found 92 companies


In [10]:
master_rows = []

for ticker in tickers:

    try:
        bs_file = BALANCE_SHEET_DIR / f"{ticker}_balancesheet.csv"
        cf_file = CASH_FLOW_DIR / f"{ticker}_cashflow.csv"
        fin_file = FINANCIALS_DIR / f"{ticker}_financials.csv"

        bs = statement_to_dict_df(bs_file, "balancesheet")
        cf = statement_to_dict_df(cf_file, "cashflow")
        fin = statement_to_dict_df(fin_file, "financials")

        company_df = (
            bs.merge(cf, on=["company", "date"], how="outer")
              .merge(fin, on=["company", "date"], how="outer")
        )

        master_rows.append(company_df)

        print(f"Processed {ticker}")

    except Exception as e:
        # companies that are missing one of the statements will be skipped
        print(f"Failed {ticker}: {e}")


Processed ANGI
Processed 002058.SZ
Processed CRM
Processed STNE
Processed MBLY
Failed DLC36116-USD: 'company'
Processed EXPE
Processed SNPS
Processed SPOT
Processed TRUE-B.ST
Processed ADSK
Processed 0GJ.F
Processed 0M5.MU
Processed IAC
Processed TTD
Failed PFIX: 'company'
Processed HP
Processed L360.F
Failed BHARATAGRI.BO: 'company'
Processed GPRO
Processed PAYO
Failed CDCETH-USD: 'company'
Processed ORCL
Failed MONTECARLO.NS: 'company'
Processed COIN
Failed 5126.T: 'company'
Processed ETOR
Processed DKNG
Processed VSCO
Failed EPIGZZX: 'company'
Failed ALTRZZX: 'company'
Failed HNS-USD: 'company'
Processed BOLT
Processed WDAY
Processed GOOG
Processed FORM
Processed TRIP
Processed CARS
Processed AMZN
Processed AVGO
Processed EBAY
Processed NET
Processed CTEV
Failed TYPTF: 'company'
Processed MELI
Processed DELL
Processed SHOP
Processed PAYC
Failed QUORZZX: 'company'
Failed UKGBBZ.XC: 'company'
Failed TTFIX: 'company'
Processed PINS
Processed 600223.SS
Processed SNAP
Processed MDT
Faile

In [17]:
import pandas as pd

dataset = pd.concat(master_rows, ignore_index=True)
dataset["date"] = pd.to_datetime(dataset["date"])

dataset["quarter"] = dataset["date"].dt.to_period("Q").astype(str)

bs_features = pd.json_normalize(
    dataset["balancesheet"]
).add_prefix("bs_")


cf_features = pd.json_normalize(
    dataset["cashflow"]
).add_prefix("cf_")

fin_features = pd.json_normalize(
    dataset["financials"]
).add_prefix("fin_")

features_df = pd.concat(
    [
        dataset[["company", "date", "quarter"]],
        bs_features,
        cf_features,
        fin_features,
    ],
    axis=1,
)
# features_df.fillna("NaN")
# print(features_df.fillna("NaN").head())

features_df.to_csv(
    MERGED_DATA["MERGED_OUTPUT_CSV_PATH"],
    index=False
)

# Verify
print(features_df.shape)
print(features_df.head())

PermissionError: [Errno 13] Permission denied: 'c:\\Users\\phuon\\Desktop\\osint-proj\\layoff-detector\\data\\processed\\merged_bs_cs_fd.csv'